In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Sample 24-hour electricity prices (currency/MWh)
hourly_prices = [100, 90, 85, 80, 75, 70, 65, 60, 55, 60, 70, 80,
                 90, 100, 110, 120, 130, 140, 135, 125, 115, 105, 95, 85]

capacity = 100          # kWh
initial_charge_level = 50  # kWh
charge_eff = 0.95
discharge_eff = 0.95


def optimize_bess(hourly_prices, capacity, initial_charge_level,
                  charge_efficiency=0.95, discharge_efficiency=0.95):
    """
    Calculates the optimal charge/discharge strategy for a Battery Energy
    Storage System (BESS) to maximise profit over a 24-hour period.

    Parameters
    ----------
    hourly_prices : list[float]
        Electricity prices for each hour of the day (currency/MWh).
        Must contain at least one value.
    capacity : float
        Maximum energy capacity of the battery (kWh). Must be > 0.
    initial_charge_level : float
        Battery state of charge at the start of the period (kWh).
        Must be in [0, capacity].
    charge_efficiency : float, optional
        Charging efficiency (0 < value <= 1). Default 0.95.
    discharge_efficiency : float, optional
        Discharging efficiency (0 < value <= 1). Default 0.95.

    Returns
    -------
    profit : float
        Net profit over the optimisation horizon.
    decisions : list[str]
        Per-hour action taken: 'Charge', 'Discharge', or 'Hold'.
    charge_levels : list[float]
        Battery state of charge after each hour (kWh).
    hourly_profits : list[float]
        Net profit contribution for each hour.
    """
    # --- Input validation (closes #5) ---
    if not hourly_prices:
        raise ValueError("hourly_prices must not be empty.")
    if capacity <= 0:
        raise ValueError("capacity must be positive.")
    if not (0 < charge_efficiency <= 1) or not (0 < discharge_efficiency <= 1):
        raise ValueError("Efficiency values must be between 0 (exclusive) and 1 (inclusive).")
    if not (0 <= initial_charge_level <= capacity):
        raise ValueError("initial_charge_level must be in [0, capacity].")

    # FIX #1 (closes #2): Compute average using np.mean()
    # Was: hourly_prices.mean  -- missing parentheses; Python lists have no .mean attribute
    avg = np.mean(hourly_prices)

    profit = 0.0
    charge_level = initial_charge_level
    decisions = []
    charge_levels = []
    hourly_profits = []

    for hour, price in enumerate(hourly_prices):
        hour_profit = 0.0

        if price < avg * 0.9 and charge_level < capacity:
            # Charge when price is below 90% of average
            charge_power = min(capacity - charge_level, 0.1 * capacity)
            charge_level += charge_power * charge_efficiency
            # FIX #3 (closes #4): Clamp to capacity; was unchecked, could exceed capacity
            charge_level = min(charge_level, capacity)
            hour_profit = -charge_power * price
            decisions.append('Charge')

        elif price > avg * 1.1 and charge_level > 0:
            # Discharge when price is above 110% of average
            discharge_power = min(charge_level, 0.1 * capacity)
            # FIX #2 (closes #3): Was charge_level -= discharge_power / discharge_efficiency
            # Dividing increases the deduction, causing negative charge levels.
            # Correct: subtract the actual discharged energy.
            charge_level -= discharge_power
            charge_level = max(charge_level, 0.0)  # guard against float underflow
            hour_profit = discharge_power * price
            decisions.append('Discharge')

        else:
            decisions.append('Hold')

        profit += hour_profit
        charge_levels.append(charge_level)
        hourly_profits.append(hour_profit)

    return profit, decisions, charge_levels, hourly_profits


# Run optimisation
total_profit, decisions, charge_levels, hourly_profits = optimize_bess(
    hourly_prices, capacity, initial_charge_level, charge_eff, discharge_eff
)

print(f"Total Profit: \u00a3{total_profit:.2f}")
print(f"Hourly Decisions: {decisions}")


In [ ]:
# Visualisations (closes #6)
hours = list(range(len(hourly_prices)))
color_map = {'Charge': 'steelblue', 'Discharge': 'tomato', 'Hold': 'lightgrey'}
bar_colors = [color_map[d] for d in decisions]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle("BESS Optimisation \u2014 24-Hour Overview", fontsize=14, fontweight="bold")

# 1. Hourly electricity prices with decision thresholds
axes[0].plot(hours, hourly_prices, marker='o', color='darkorange', linewidth=2)
axes[0].axhline(np.mean(hourly_prices), color='grey', linestyle='--', label='Average price')
axes[0].axhline(np.mean(hourly_prices) * 0.9, color='steelblue', linestyle=':', label='Charge threshold (\u00d70.9)')
axes[0].axhline(np.mean(hourly_prices) * 1.1, color='tomato', linestyle=':', label='Discharge threshold (\u00d71.1)')
axes[0].set_ylabel("Price (\u00a3/MWh)")
axes[0].set_title("Electricity Prices")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# 2. Battery state of charge over the day
axes[1].fill_between(hours, charge_levels, alpha=0.4, color='green')
axes[1].plot(hours, charge_levels, marker='s', color='green', linewidth=2)
axes[1].axhline(capacity, color='red', linestyle='--', label=f'Capacity ({capacity} kWh)')
axes[1].set_ylabel("State of Charge (kWh)")
axes[1].set_title("Battery State of Charge")
axes[1].set_ylim(0, capacity * 1.1)
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# 3. Hourly profit coloured by decision type
axes[2].bar(hours, hourly_profits, color=bar_colors, edgecolor='white')
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_xlabel("Hour of Day")
axes[2].set_ylabel("Profit (\u00a3)")
axes[2].set_title("Hourly Profit / Loss")
axes[2].set_xticks(hours)
legend_patches = [mpatches.Patch(color=c, label=l) for l, c in color_map.items()]
axes[2].legend(handles=legend_patches, fontsize=8)
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('bess_optimisation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nChart saved to bess_optimisation_results.png")
